<b>Note: </b>The algorithms used to create machine learning using Tree Based models include Random Forest Regression, XGBoost, and LightGBM. So the features do not need to be scaled or standardized.

All configurations are set to the best parametric.

1. Import library

In [1]:
import pandas as pd
import numpy as np
import os
import joblib

from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV

from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

2. Read clean_data_oil.xlsx and sort the values

In [2]:
df = pd.read_excel('clean_data_oil.xlsx')
df = df.sort_values(['country', 'year']).reset_index(drop=True)

3. Encoding country feature, convert category (country) data into numbers based on target averages

In [3]:
country_mean = df.groupby('country')['oil_consumption'].mean()
df['country_te'] = df['country'].map(country_mean)

4. Feature Engineering (LAG + ROLLING) with the last 3 years period

In [4]:
lags = [1, 2, 3]

for lag in lags:
    df[f'lag_{lag}'] = df.groupby('country')['oil_consumption'].shift(lag)

# rolling mean
df['rolling_mean_3'] = (
    df.groupby('country')['oil_consumption']
    .shift(1)
    .rolling(3)
    .mean()
)

# drop NA
df = df.dropna().reset_index(drop=True)

5. Define Features with lags

In [5]:
features = [
    'year',
    'population',
    'gdp',
    'oil_electricity',
    'oil_elec_per_capita',
    'oil_prod_per_capita',
    'oil_production',
    'country_te',
    'rolling_mean_3'
] + [f'lag_{lag}' for lag in lags]

target = 'oil_consumption'

6. Time-based split, train 1974-2022 test 2023-2024

In [6]:
train = df[df['year'] <= df['year'].max() - 2]
test  = df[df['year'] > df['year'].max() - 2]

X_train, y_train = train[features], train[target]
X_test, y_test   = test[features], test[target]

7. Train Model Random Forest Regressor

In [7]:
tscv = TimeSeriesSplit(n_splits=3)
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_param = {
    'n_estimators': [200, 300, 500, 800, 1000],
    'max_depth': [8, 12, 16, None],
    'min_samples_leaf': [1, 3, 5, 7],
    'min_samples_split': [2, 5, 10, 15],
    'max_features': ['sqrt', 0.8]
}

rf_search = RandomizedSearchCV(
    rf,
    rf_param,
    n_iter=45,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

rf_search.fit(X_train, y_train)

rf_best = rf_search.best_estimator_
print(f"Best RF Params: {rf_search.best_params_}")

Fitting 3 folds for each of 45 candidates, totalling 135 fits
Best RF Params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.8, 'max_depth': None}


8. Train Model XGBoost

In [8]:
xgb = XGBRegressor(
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_param = {
    'n_estimators': [300, 500, 800, 1000],
    'max_depth': [4, 6, 8, 16],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5, 7]
}

xgb_search = RandomizedSearchCV(
    xgb,
    xgb_param,
    n_iter=45,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

xgb_search.fit(X_train, y_train)

xgb_best = xgb_search.best_estimator_
print(f"Best XGB Params: {xgb_search.best_params_}")

Fitting 3 folds for each of 45 candidates, totalling 135 fits
Best XGB Params: {'subsample': 1.0, 'n_estimators': 1000, 'min_child_weight': 7, 'max_depth': 16, 'learning_rate': 0.01, 'colsample_bytree': 0.8}


9. Train Model LightGBM

In [9]:
tscv = TimeSeriesSplit(n_splits=3)

lgb = LGBMRegressor(
    random_state=42,
    n_jobs=-1
)

lgb_param = {
    'n_estimators': [300, 500, 800, 1000],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [-1, 5, 7, 16],
    'num_leaves': [31, 50, 70],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_samples': [10, 20, 30, 40]
}

lgb_search = RandomizedSearchCV(
    estimator=lgb,
    param_distributions=lgb_param,
    n_iter=45,
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1,
    random_state=42
)

lgb_search.fit(X_train, y_train)

lgb_best = lgb_search.best_estimator_

print(f"Best Params: {lgb_search.best_params_}")

Fitting 3 folds for each of 45 candidates, totalling 135 fits
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004776 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2677
[LightGBM] [Info] Number of data points in the train set: 9153, number of used features: 12
[LightGBM] [Info] Start training from score 200.432357
Best Params: {'subsample': 1.0, 'num_leaves': 50, 'n_estimators': 1000, 'min_child_samples': 10, 'max_depth': -1, 'learning_rate': 0.1, 'colsample_bytree': 0.7}


10. Evaluation all Model

In [10]:
models = {
    "RandomForest": rf_best,
    "XGBoost": xgb_best,
    "LightGBM": lgb_best,
}

for name, m in models.items():
    pred = m.predict(X_test)

    mae = mean_absolute_error(y_test, pred)
    rmse = mean_squared_error(y_test, pred, squared=False)
    r2 = r2_score(y_test, pred)

    print(f"\n{name}")
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R2   : {r2:.4f}")


RandomForest
MAE  : 12.6035
RMSE : 57.3731
R2   : 0.9972

XGBoost
MAE  : 11.8806
RMSE : 61.2437
R2   : 0.9968

LightGBM
MAE  : 12.2393
RMSE : 43.1317
R2   : 0.9984


d:\Programmer\Python\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
d:\Programmer\Python\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(
d:\Programmer\Python\Lib\site-packages\sklearn\metrics\_regression.py:483: FutureWarning: 'squared' is deprecated in version 1.4 and will be removed in 1.6. To calculate the root mean squared error, use the function'root_mean_squared_error'.
  warnings.warn(


11. Prepare last Data per country

In [11]:
last_data = df.groupby('country').tail(1).copy()

future_years = [2025, 2026]

12. Forecast for year 2025-2026 by using the LightGBM model because this model is the best

In [12]:
results = []

for idx, row in last_data.iterrows():

    current = row.copy()
    country_name = current['country']

    for year in future_years:

        current['year'] = year

        # prediction
        X_pred = current[features].values.reshape(1, -1)

        pred = lgb_best.predict(X_pred)[0]

        # no negative
        pred = max(pred, 0)

        if pred < 1:
            pred = 0

        # save result
        results.append({
            'country': country_name,
            'year': year,
            'predicted_oil_consumption': pred,
            'Type': 'Forecast'
        })

        # update lag recursive
        for lag in reversed(lags):

            if lag == 1:
                current['lag_1'] = pred

            else:
                current[f'lag_{lag}'] = current[f'lag_{lag-1}']

        # rolling mean update
        lag_values = [current[f'lag_{i}'] for i in lags]

        current['rolling_mean_3'] = np.mean(lag_values)

# make dataframe
future_df = pd.DataFrame(results)

# drop duplicate (if there are)
future_df = future_df.drop_duplicates(
    subset=['country', 'year'],
    keep='first'
)

# reset index
future_df = future_df.reset_index(drop=True)

print(future_df)

         country  year  predicted_oil_consumption      Type
0    Afghanistan  2025                   0.000000  Forecast
1    Afghanistan  2026                   0.000000  Forecast
2        Albania  2025                   0.000000  Forecast
3        Albania  2026                   0.000000  Forecast
4        Algeria  2025                 247.683855  Forecast
..           ...   ...                        ...       ...
435        Yemen  2026                   2.245873  Forecast
436       Zambia  2025                   0.000000  Forecast
437       Zambia  2026                   0.000000  Forecast
438     Zimbabwe  2025                   0.000000  Forecast
439     Zimbabwe  2026                   0.000000  Forecast

[440 rows x 4 columns]


13. Show the forecast results

In [13]:
future_df = pd.DataFrame(results)

future_df.loc[future_df['predicted_oil_consumption'] < 1, 'predicted_oil_consumption'] = 0

print(future_df)

         country  year  predicted_oil_consumption      Type
0    Afghanistan  2025                   0.000000  Forecast
1    Afghanistan  2026                   0.000000  Forecast
2        Albania  2025                   0.000000  Forecast
3        Albania  2026                   0.000000  Forecast
4        Algeria  2025                 247.683855  Forecast
..           ...   ...                        ...       ...
435        Yemen  2026                   2.245873  Forecast
436       Zambia  2025                   0.000000  Forecast
437       Zambia  2026                   0.000000  Forecast
438     Zimbabwe  2025                   0.000000  Forecast
439     Zimbabwe  2026                   0.000000  Forecast

[440 rows x 4 columns]


14. Save Forecast & Models

In [14]:
os.makedirs('Forecast', exist_ok=True)
os.makedirs('Models', exist_ok=True)

# save to excel
future_df.to_excel('Forecast/oil_forecasts_2025-2026.xlsx', index=False)
print("Forecasts saved to 'Forecast/oil_forecasts_2025-2026.xlsx'")

# save best model
joblib.dump(rf_best, 'Models/randomforest.pkl')
joblib.dump(xgb_best, 'Models/xgboost.pkl')
joblib.dump(lgb_best, 'Models/lightgbm.pkl')

print("Models saved to 'Models/' directory")

Forecasts saved to 'Forecast/oil_forecasts_2025-2026.xlsx'
Models saved to 'Models/' directory
